In [2]:
class FitPlane:
    def solve_equations(X, y):
        """
        Solve the linear system X·β = y via Gaussian elimination (with partial pivoting).

        Parameters
        ----------
        X : list of list of floats
            Coefficient matrix of shape (n, n).
        y : list of floats
            Right‐hand‐side vector of length n.

        Returns
        -------
        beta : list of floats
            Solution vector β such that X·β = y.

        Raises
        ------
        ValueError
            If the matrix is singular or input sizes are inconsistent.
        """
        # Number of equations
        n = len(X)
        if any(len(row) != (n) for row in X):
            raise ValueError(f"All rows of X must have length {n}")
        if len(y) != n:
            raise ValueError(f"Length of y must be {n}")

        # Build augmented matrix A = [X | y]
        # and convert all entries to float
        A = [list(map(float, X[i])) + [float(y[i])] for i in range(n)]

        # Forward elimination
        for i in range(n):
            # Partial pivoting: find row with max abs value in column i
            pivot_row = max(range(i, n), key=lambda r: abs(A[r][i]))
            if abs(A[pivot_row][i]) < 1e-12:
                raise ValueError("Matrix is singular or nearly singular")
            # Swap current row with pivot_row
            A[i], A[pivot_row] = A[pivot_row], A[i]

            # Eliminate entries below A[i][i]
            for r in range(i + 1, n):
                factor = A[r][i] / A[i][i]
                # subtract factor * row i from row r
                for c in range(i, n + 1):
                    A[r][c] -= factor * A[i][c]

        # Back substitution
        beta = [0.0] * n
        for i in range(n - 1, -1, -1):
            rhs = A[i][n]  # the augmented value
            for j in range(i + 1, n):
                rhs -= A[i][j] * beta[j]
            beta[i] = rhs / A[i][i]

        return beta
    def fit(self, X, y):
        # X will be records of x1, x2, x3....
        # X will be a 2D array - [[1,1], [3,4], [6, 7]] - matrix
        # y will be 1D array - [2, 4, 5] - vector
        # [[1,1, 1, 2]]
        # [[3, 4, 1, 4]]
        # [[]]
        self.parameters = FitPlane.solve_equations(X, y)
    def predict(self,X):
        result = []
        for x in X:
            y = 0
            for i in range(len(x)):
                y += x[i] * self.parameters[i]
            result.append(y)
        return result
    

In [3]:
model = FitPlane()
model.fit([[1,2],[4,5]], [4, 7])
model.predict([[7,8]])

[10.0]

In [4]:
# Tree
class Node:
    val = 0
    left = None
    right = None

    def insert(self, head,v):
        if head == None:
            return Node(v)
        if v < head.val:
            head.left = self.insert(head.left, v)
        else:
            head.right = self.insert(head.right, v)
        return head

    def __init__(self, v, l = None, r = None):
        self.val = v
        self.left = l
        self.right = r
    
    def total(self):
        ret = self.val
        if self.left != None:
            ret += self.left.total()
        if self.right != None:
            ret += self.right.total()
        return ret
    def count(self):
        cnt = 1
        if self.left != None:
            cnt += self.left.count()
        if self.right != None:
            cnt += self.right.count()
        return cnt
    def min(self):
        min = self.val
        if self.left != None:
            min = self.left.min()
        return min
    def max(self):
        max = self.val
        if self.right != None:
            max = self.right.max()
        return max
    def find(self,x):
        if self.val == x:
            return True
        if x<self.val:
            if self.left != None:
                return self.left.find(x)
            else:
                return False
        if x>self.val:
            if self.right != None:
                return self.right.find(x)
            else:
                return False
    def translate_update_value(self, func):
        self.val = func(self.val)
        if self.left != None:
            self.left.translate_update_value(func)
        if self.right != None:
            self.right.translate_update_value(func)
    def max_depth(self):
        lef_depth = 0
        rig_depth = 0
        if self.left != None:
            lef_depth += self.left.max_depth()
        if self.right != None:
            rig_depth += self.right.max_depth()
        if lef_depth > rig_depth:
            return lef_depth + 1
        else:
            return rig_depth + 1


In [8]:
#Create binary search tree
notes_to_insert = [5, 3, 7, 2, 4, 6, 8]
head = None
for note in notes_to_insert:
    if head == None:
        head = Node(note)
    else:
        head.insert(head, note)


#Print the tree
def print_tree(node, level=0):
    if node is not None:
        print_tree(node.right, level + 1)
        print(' ' * 4 * level + '->', node.val)
        print_tree(node.left, level + 1)
print_tree(head)


print(head.total())
print(head.count())
print(head.min())
print(head.max())
print(head.find(5))
print(head.find(10))
print(head.max_depth())
head.translate_update_value(lambda x: x + 1)
print_tree(head)



        -> 8
    -> 7
        -> 6
-> 5
        -> 4
    -> 3
        -> 2
35
7
2
8
True
False
3
        -> 9
    -> 8
        -> 7
-> 6
        -> 5
    -> 4
        -> 3


Multi level decisin tree

In [10]:
class DecisionNode:
    def __init__(self, criteria, boundary, left, right, operator=">="):
        self.criteria = criteria
        self.boundary = boundary
        self.left = left
        self.right = right
        self.operator = operator  # can be ">=" or "<"

    def answer(self, circumstance):
        value = circumstance[self.criteria]

        # Handle custom comparison logic
        if self.operator == ">=":
            decision = value >= self.boundary
        elif self.operator == "<":
            decision = value < self.boundary
        else:
            raise ValueError(f"Unsupported operator: {self.operator}")

        return self.left.answer(circumstance) if decision else self.right.answer(circumstance)


class Yes(DecisionNode):
    def __init__(self):
        pass

    def answer(self, circumstance=None):
        return True


class No(DecisionNode):
    def __init__(self):
        pass

    def answer(self, circumstance=None):
        return False


In [13]:
dist = DecisionNode("distance", 1, Yes(), No(), operator="<")
learning_opp = DecisionNode("learning_opportunity", 2, dist, No(), operator=">=")
salary = DecisionNode("salary", 1, learning_opp,No(), operator=">=")
# Test Case 1 — Should return True
print(salary.answer({
    'salary': 3,
    'learning_opportunity': 3,
    'distance': 0.5
}))  # ✅ True

# Test Case 2 — Should return False
print(salary.answer({
    'salary': 1.5,
    'learning_opportunity': 3,
    'distance': 2
}))  # ❌ False

# Test Case 3 — Should return False
salary.answer({
    'salary': 0.5,
    'learning_opportunity': 3,
    'distance': 0.5
})

True
False


False